# Pulling Compustat Annual Fundamentals and the CCM Link Table

## What this notebook does and why

The Kelly dataset provides pre-computed signals up to December 2021. For the 2022–2024 extension period, we need to construct those same signals from scratch using raw accounting data. That requires two things from Compustat: the annual fundamentals (balance sheet, income statement, cash flow items) and the quarterly fundamentals (for higher-frequency signals like quarterly ROA and accrual volatility).

Before we can join Compustat to CRSP, we also need the CCM link table — the bridge that maps CRSP permnos to Compustat gvkeys. These are two separate identifier systems and there is no direct way to link them without this table.

This notebook produces three files:
- `link_table.csv` — the permno ↔ gvkey mapping for our universe
- `compustat_annual.csv` — annual fundamentals for all firms in the extension universe, starting from 1950 to provide full lagged variable history

The quarterly Compustat pull (`compustat_quarterly.csv`) is handled in the extension notebook.

## Why we start from 1950 even though the extension is 2022–2024

Several of our features require lagged accounting data. Asset growth (`agr`) needs the prior year's total assets. The Piotroski signals need year-over-year comparisons. If we only pulled data from 2022, the first year of extension features would have no history to compute these lags and would be entirely NaN. Starting from 1950 gives every firm a full multi-decade history so that lag-based features are fully populated by 2022.

## The CCM link table — why the quality filters matter

Not all links in the CCM table are equal. CRSP maintains a quality hierarchy:

- **linktype LC and LU**: confirmed and unconfirmed primary links — the most reliable. LC means the link has been manually confirmed. LU means CRSP is confident but it has not been manually verified.
- **linkprim P and C**: primary and primary calendar links — meaning this is the main listing for this company at this time, not a secondary or historical cross-listing.

Using lower-quality link types (LS, LN, etc.) would introduce ambiguous matches where a single permno maps to multiple gvkeys simultaneously, or where the link covers a period when the companies were not actually the same entity. We only use LC and LU / P and C to avoid this.

We also fill missing `linkenddt` with a far-future date (2099-12-31) so that the date-validity filter works correctly for firms that are still active today and have no recorded end date.

## Step 1 — Connect to WRDS and verify CRSP and Compustat data availability

Before pulling anything I check the most recent dates available in both CRSP and Compustat. This confirms that the databases have been updated to cover the extension period and tells me exactly how far I can pull.

In [ ]:
import wrds
conn = wrds.Connection(wrds_username='muhab')

# Check the last available date in CRSP before pulling
# This confirms the database is current and tells us how far the extension can go
latest = conn.raw_sql("""
    SELECT MAX(date) as last_date
    FROM crsp.msf
""")
print("Last date in CRSP:", latest['last_date'].values[0])

## Step 2 — Identify the extension universe

The extension period is January 2022 to November 2024. I extract the unique permnos that appear in CRSP during this window — these are the stocks we need to construct features for. This becomes the filter for everything pulled from Compustat: no point pulling accounting data for firms that are not in our trading universe.

In [ ]:
import pandas as pd

# Load the cleaned CRSP file to identify which stocks appear in the extension period
crsp = pd.read_csv('crsp_returns.csv', parse_dates=['date'])

# The extension period starts where the Kelly dataset ends
extension = crsp[crsp['date'] >= '2022-01-01']
print(f"Extension period rows: {len(extension):,}")
print(f"Date range: {extension['date'].min()} to {extension['date'].max()}")
print(f"Unique permnos: {extension['permno'].nunique():,}")

## Step 3 — Check Compustat coverage

Before pulling, verify that Compustat is sufficiently current to cover the extension period. The last date in Compustat should be at least 6 months ahead of our last CRSP observation, since annual data requires a 6-month lag before it is considered publicly available.

In [ ]:
# Verify Compustat is current enough to cover our extension period
# With a 6-month availability lag, Compustat data from mid-2024 covers 2025 portfolio dates
latest_comp = conn.raw_sql("""
    SELECT MAX(datadate) as last_date
    FROM comp.funda
    WHERE indfmt = 'INDL'
    AND datafmt = 'STD'
    AND popsrc = 'D'
    AND consol = 'C'
""")
print("Last Compustat date:", latest_comp['last_date'].values[0])

## Step 4 — Pull the CCM link table

The CCM (CRSP/Compustat Merged) link table maps CRSP permnos to Compustat gvkeys. Without this, there is no way to join the two databases since they use entirely different identifier systems.

I filter to only the highest-quality link types (LC, LU) and primary link designations (P, C). I also apply a date-validity filter: links have start and end dates, and we only want links that were valid during the period we care about. The `linkenddt` is filled with 2099-12-31 for active firms that have no recorded end date — this ensures the date filter does not incorrectly exclude current firms.

In [ ]:
# Get the permnos active in our extension period — these drive every subsequent pull
extension_permnos = crsp[crsp['date'] >= '2022-01-01']['permno'].unique().tolist()
print(f"Permnos in extension period: {len(extension_permnos):,}")

def pull_link_table(conn, permnos):
    permno_str = ','.join(map(str, permnos))

    print("Pulling CCM link table...")
    link = conn.raw_sql(f"""
        SELECT
            gvkey,
            lpermno AS permno,
            linktype,
            linkprim,
            linkdt,
            linkenddt
        FROM crsp.ccmxpf_lnkhist
        WHERE linktype IN ('LC', 'LU')
            AND linkprim IN ('P', 'C')
            AND lpermno IN ({permno_str})
    """, date_cols=['linkdt', 'linkenddt'])

    # Fill missing end dates with a far-future date
    # Active firms have no linkenddt — without this fill, they would be
    # incorrectly excluded by any date-range filter applied downstream
    link['linkenddt'] = link['linkenddt'].fillna(pd.Timestamp('2099-12-31'))

    print(f"Link rows: {len(link):,}")
    print(f"Unique gvkeys: {link['gvkey'].nunique():,}")
    print(f"Unique permnos matched: {link['permno'].nunique():,}")

    return link

link = pull_link_table(conn, extension_permnos)
link.to_csv('link_table.csv', index=False)
print("Saved: link_table.csv")

## Step 5 — Pull Compustat annual fundamentals

With the gvkeys from the link table, I pull annual fundamentals from `comp.funda`. The Compustat filters applied in the SQL query are standard and important:

- **`indfmt = 'INDL'`** — industrial format only, excludes financial firms which report fundamentals on a very different structure
- **`datafmt = 'STD'`** — standardised format, ensures consistent variable definitions across firms and time
- **`popsrc = 'D'`** — domestic firms only
- **`consol = 'C'`** — consolidated statements only, so we get the full consolidated entity not individual subsidiaries
- **`at > 0`** — removes firms with zero or negative total assets which indicate data errors

The pull is chunked to avoid WRDS query timeouts. 

Duplicate `(gvkey, datadate)` pairs are removed by keeping the last observation — Compustat occasionally has restated entries and we want the most recent version.

In [ ]:
# Re-pull with txt included — this was missing from the first pull
# txt (income tax expense) is required for the tax-to-book signal

import wrds
import pandas as pd

conn = wrds.Connection(wrds_username='muhab')

link = pd.read_csv('link_table.csv')
# Standardise gvkey format: string, stripped, zero-padded to 6 digits
# Compustat and CCM can have inconsistent formatting — this ensures the join works
link['gvkey'] = link['gvkey'].astype(str).str.strip().str.zfill(6)
gvkeys = link['gvkey'].unique().tolist()

chunk_size = 500
chunks     = [gvkeys[i:i+chunk_size] for i in range(0, len(gvkeys), chunk_size)]

all_chunks = []
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}/{len(chunks)}...")
    gvkey_str = ','.join([f"'{g}'" for g in chunk])

    q = conn.raw_sql(f"""
        SELECT
            gvkey, datadate, fyear, fyr,
            -- Balance sheet items
            at, lt, seq, ceq,
            pstk, pstkrv, pstkl,
            txditc, txp,
            act, lct, che, rect, invt,
            dlc, dltt,
            -- Income statement
            ni, ib, oiadp, oibdp,
            dp, sale, cogs, xsga, xrd,
            -- Cash flow
            capx,
            -- Other items needed for signal construction
            dv, csho, prcc_f,
            naicsh, sich
        FROM comp.funda
        WHERE datadate BETWEEN '1950-01-01' AND '2024-12-31'
            AND gvkey IN ({gvkey_str})
            AND indfmt = 'INDL'
            AND datafmt = 'STD'
            AND popsrc = 'D'
            AND consol = 'C'
            AND at > 0
    """, date_cols=['datadate'])

    all_chunks.append(q)

comp = pd.concat(all_chunks, ignore_index=True)

# Remove duplicate (gvkey, datadate) pairs — keep the most recent entry
# Compustat sometimes has restated entries and we want the latest version
comp = comp.sort_values(['gvkey', 'datadate'])
comp = comp.drop_duplicates(subset=['gvkey', 'datadate'], keep='last')

print(f"Rows: {len(comp):,}")
print(f"txt coverage: {comp['txt'].notna().mean():.1%}")

comp.to_csv('compustat_annual.csv', index=False)
print("Saved: compustat_annual.csv")
conn.close()